# 03 LABELS: checks on the episodes and labels

Audits the labelling used in the paper and prepares the label variants for the sensitivity analysis. Nothing upstream is changed.

1. **Unmatched onsets.** Reproduce the episode pairing with a reason code for every onset left without a recovery, and count them over the full log and in June 2025.
2. **Exclusion rule.** Show what the in-episode exclusion does to unmatched (zero-duration) episodes, and confirm that the stored labels are reproduced.
3. **Window definitions.** Confirm that the windows are in seconds, that each warning window ends one second before onset, and that D5 keeps its post-onset seconds.
4. **Label variants** for `10_UNMATCHED`: (A) unmatched onsets removed; (B) unmatched episodes given the median matched duration of their message.
5. **July labels** for `19_AGEING`: 1 to 12 July 2025, labelled under the same rules.

Inputs: `DATA/37011FLv4.csv`, `build/fault_episodes_v4.csv`, `build/telemetry_labelled_v4.pkl`, `../ANALYSIS2/TELEMETRY/build/telemetry_raw_v2.pkl`. Outputs in `OUTPUT/v4/` and `build/`.

Paper: Sections 2.5 and 2.7.1.

In [ ]:
import os, sys, gc
import numpy as np, pandas as pd

A2 = "../ANALYSIS2"
CFG = dict(
    fault_log   = os.environ.get("REV_DATADIR", "DATA") + "/37011FLv4.csv",
    episodes_v2 = os.environ.get("REV_BUILDDIR", "build") + "/fault_episodes_v4.csv",
    labelled_v2 = os.environ.get("REV_BUILDDIR", "build") + "/telemetry_labelled_v4.pkl",
    raw_v2      = f"{A2}/TELEMETRY/build/telemetry_raw_v2.pkl",
    outdir      = os.environ.get("REV_OUTDIR", "OUTPUT") + "/v4",
    builddir    = os.environ.get("REV_BUILDDIR", "build"),
    june        = ("2025-06-01 00:00:00", "2025-06-30 23:59:59"),
    july        = ("2025-07-01 00:00:00", "2025-07-12 23:59:59"),
    max_episode_s = 6 * 3600,
    debounce_s    = 60,
    blower      = "AUX CAP MACH RM BLOW1 NOT OFF",
    horizons    = {"D5": (5, 5, "detection"), "W5": (5, -1, "warning"),
                   "W60": (60, -1, "warning"), "W300": (300, -1, "warning")},
)
os.makedirs(CFG["outdir"], exist_ok=True)
os.makedirs(CFG["builddir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
SUMMARY = {}

## 1. Episode pairing with reasons for unmatched onsets

`build_episodes` is the pairing function used for the episodes, with one addition: a `reason` column. A recovery is accepted only if it is the next unused recovery of the same fault in the same group, falls no earlier than the onset second, no later than the next onset of that fault, and within the six-hour cap. Otherwise the onset is unmatched, and its recovery is set equal to its onset, giving zero duration.

In [ ]:
def build_episodes(cfg):
    fl = pd.read_csv(cfg["fault_log"])
    fl["dt"] = pd.to_datetime(fl["DATETIME"], format="%m/%d/%Y %H:%M:%S")
    fl["is_rec"] = fl["FAULT"].str.contains("Recovered", case=False, na=False)
    fl["base"] = (fl["FAULT"].str.replace(" Recovered", "", regex=False)
                             .str.strip().str.upper())
    fl = fl.sort_values("dt").reset_index(drop=True)
    episodes = []
    for (cluster, base), grp in fl.groupby(["CLUSTER", "base"]):
        onsets = grp.loc[~grp.is_rec, "dt"].tolist()
        recovs = grp.loc[grp.is_rec, "dt"].tolist()
        ri = 0
        for k, on in enumerate(onsets):
            nxt = onsets[k + 1] if k + 1 < len(onsets) else pd.Timestamp.max
            while ri < len(recovs) and recovs[ri] < on:
                ri += 1
            end, dur, matched, reason = on, 0.0, False, "no later recovery in log"
            if ri < len(recovs):
                cand = recovs[ri]
                d = (cand - on).total_seconds()
                if cand > nxt:
                    reason = "next onset of the same fault comes first"
                elif d > cfg["max_episode_s"]:
                    reason = "recovery beyond the 6 h cap"
                else:
                    end, dur, matched, reason = cand, d, True, "matched"
                    ri += 1
            episodes.append(dict(cluster=cluster, fault=base, onset=on, recovery=end,
                                 duration_s=dur, matched=matched, reason=reason))
    ep = pd.DataFrame(episodes).sort_values("onset").reset_index(drop=True)
    ep["debounce_keep"] = True
    for c, g in ep.groupby("cluster"):
        gaps = g["onset"].diff().dt.total_seconds()
        ep.loc[g.index, "debounce_keep"] = gaps.isna() | (gaps > cfg["debounce_s"])
    return ep

ep = build_episodes(CFG)
ep_v2 = pd.read_csv(CFG["episodes_v2"], parse_dates=["onset", "recovery"])
cols = ["cluster", "fault", "onset", "recovery", "duration_s", "matched", "debounce_keep"]
a = ep[cols].sort_values(["onset", "cluster", "fault"]).reset_index(drop=True)
b = ep_v2[cols].sort_values(["onset", "cluster", "fault"]).reset_index(drop=True)
same = a.astype(str).equals(b.astype(str))
print("Episodes reproduce fault_episodes_v4.csv:", "OK" if same else "DIFFERENT")
assert same, "pairing does not reproduce the stored episodes; stop and investigate"

ep["group"] = np.where(ep.fault.eq(CFG["blower"]), "blower", "other")
um = ep[~ep.matched]
print(f"Unmatched, zero duration: {len(um)} of {len(ep)} episodes "
      f"({100*len(um)/len(ep):.1f}%); all have recovery == onset: {bool((um.recovery == um.onset).all())}")

In [ ]:
j0, j1 = (pd.Timestamp(x) for x in CFG["june"])
ep["june"] = ep.onset.between(j0, j1)

def tab(df):
    t = (df.groupby(["reason", "group"]).size().unstack(fill_value=0)
           .reindex(columns=["blower", "other"], fill_value=0))
    t["total"] = t.sum(axis=1)
    t.loc["all onsets"] = df.groupby("group").size().reindex(["blower", "other"], fill_value=0).tolist() + [len(df)]
    return t

full_t, june_t = tab(ep), tab(ep[ep.june])
print("FULL LOG, 4 Nov 2024 to 21 Jul 2025\n", full_t.to_string(), "\n")
print("JUNE 2025\n", june_t.to_string())
pd.concat({"full log": full_t, "June": june_t}).to_csv(f"{CFG['outdir']}/rev_unmatched.csv")

SUMMARY.update(
    unmatched_full=int((~ep.matched).sum()), episodes_full=len(ep),
    unmatched_june=int((~ep[ep.june].matched).sum()), episodes_june=int(ep.june.sum()),
    unmatched_june_blower=int((~ep[ep.june & ep.group.eq("blower")].matched).sum()),
    unmatched_june_other=int((~ep[ep.june & ep.group.eq("other")].matched).sum()),
)

In [ ]:
# Which labels go unmatched, and does the label ever log a recovery at all?
has_rec = ep.groupby("fault")["matched"].any().rename("label_ever_matched")
lab = (ep[~ep.matched].groupby(["fault", "reason"]).size().rename("unmatched").reset_index()
         .merge(ep.groupby("fault").size().rename("onsets").reset_index(), on="fault")
         .merge(has_rec.reset_index(), on="fault")
         .sort_values("unmatched", ascending=False))
print(lab.head(15).to_string(index=False))
never = ep[~ep.fault.map(has_rec)]
print(f"\nLabels that never log a matched recovery: {never.fault.nunique()}, "
      f"accounting for {len(never)} of the {int((~ep.matched).sum())} unmatched onsets")
lab.to_csv(f"{CFG['outdir']}/rev_unmatched_by_label.csv", index=False)
SUMMARY.update(unmatched_in_never_recovered_labels=len(never))

## 2. What the in-episode exclusion does to unmatched episodes

`label` is the labelling function used for the paper. A second is excluded (-1) if it lies between an onset and its recovery, inclusive. An unmatched episode has recovery equal to onset, so it excludes its onset second only. For the warning windows exclusion takes precedence over the window; for D5 the window takes precedence. The stored labels are recomputed here first, so that every count below refers to the labels used in `07_CLASSIFY`.

In [ ]:
ONE_S = np.timedelta64(1, "s")

def label(index, ep, pre_s, post_s, kind):
    n = len(index)
    win = np.zeros(n, dtype=bool)
    epi = np.zeros(n, dtype=bool)
    lo = index.searchsorted
    for on, rec in zip(ep["onset"].values, ep["recovery"].values):
        win[lo(on - pre_s * ONE_S, "left"):lo(on + post_s * ONE_S, "right")] = True
        epi[lo(on, "left"):lo(max(rec, on), "right")] = True
    y = np.zeros(n, dtype=np.int8)
    if kind == "detection":
        y[epi] = -1
        y[win] = 1
    else:
        y[win] = 1
        y[epi] = -1
    return y, int((win & epi).sum())

def episode_mask(index, ep):
    m = np.zeros(len(index), dtype=bool)
    lo = index.searchsorted
    for on, rec in zip(ep["onset"].values, ep["recovery"].values):
        m[lo(on, "left"):lo(max(rec, on), "right")] = True
    return m

tel = pd.read_pickle(CFG["labelled_v2"]).sort_values("DATETIME").reset_index(drop=True)
idx = tel["DATETIME"].values
span = ep[ep.june]

rows = []
for h, (pre, post, kind) in CFG["horizons"].items():
    y, _ = label(idx, span, pre, post, kind)
    rows.append(dict(label=h, reproduces_stored=bool(np.array_equal(y, tel[h].values))))
repro = pd.DataFrame(rows).set_index("label")
print(repro.to_string())
assert repro.reproduces_stored.all(), "stored labels not reproduced; stop and investigate"

m_all = episode_mask(idx, span)
m_matched = episode_mask(idx, span[span.matched])
m_unmatched = episode_mask(idx, span[~span.matched])
only_unmatched = m_unmatched & ~m_matched
present = int(m_unmatched.sum())
print(f"\nJune unmatched onsets: {(~span.matched).sum()}; onset seconds present in telemetry: {present}")
print(f"Seconds excluded only because of an unmatched onset: {int(only_unmatched.sum())} "
      f"of {int(m_all.sum()):,} excluded seconds")

rows = []
for h, (pre, post, kind) in CFG["horizons"].items():
    # window seconds before any exclusion
    w = np.zeros(len(idx), dtype=bool)
    for on in span.onset.values:
        w[idx.searchsorted(on - pre * ONE_S, "left"):idx.searchsorted(on + post * ONE_S, "right")] = True
    rows.append(dict(label=h, kind=kind,
                     window_seconds=int(w.sum()),
                     inside_any_episode=int((w & m_all).sum()),
                     inside_unmatched_only=int((w & only_unmatched).sum()),
                     treatment="kept positive" if kind == "detection" else "excluded",
                     positive=int((tel[h] == 1).sum()), excluded=int((tel[h] == -1).sum())))
excl = pd.DataFrame(rows).set_index("label")
print("\n", excl.to_string())
excl.to_csv(f"{CFG['outdir']}/rev_exclusion_audit.csv")
SUMMARY.update(unmatched_onset_seconds_present=present,
               excluded_only_by_unmatched=int(only_unmatched.sum()),
               excluded_total_june=int(m_all.sum()),
               D5_positives_inside_episodes=int(excl.loc["D5", "inside_any_episode"]))

## 3. Window definitions

For each label, the runs of consecutive positive seconds are measured. Isolated windows
should run for 5, 60 and 300 seconds at W5, W60 and W300, and 11 seconds at D5 (five
before, the onset second, five after). Every warning window must end before its onset
second, and the onset second must never be positive at a warning horizon.

In [ ]:
ts = pd.DatetimeIndex(tel["DATETIME"].values)
pos_of = pd.Series(np.arange(len(tel)), index=ts)
onset_rows = pos_of.reindex(pd.DatetimeIndex(span.onset.values)).dropna().astype(int).values

def runs(y):
    p = (y == 1).astype(np.int8)
    d = np.diff(np.r_[0, p, 0])
    starts, ends = np.where(d == 1)[0], np.where(d == -1)[0]
    # a run must also be contiguous in time, not only in row order
    out = []
    t = tel["DATETIME"].values
    for s, e in zip(starts, ends):
        seg = t[s:e]
        gaps = np.where(np.diff(seg) != ONE_S)[0]
        bounds = np.r_[0, gaps + 1, len(seg)]
        out += list(np.diff(bounds))
    return np.array(out)

rows = []
for h, (pre, post, kind) in CFG["horizons"].items():
    r = runs(tel[h].values)
    expected = pre + post + 1 if kind == "detection" else pre
    mode = pd.Series(r).mode().iloc[0]
    onset_positive = float(np.mean(tel[h].values[onset_rows] == 1))
    rows.append(dict(label=h, expected_run_s=expected, modal_run_s=int(mode),
                     runs=len(r), share_at_expected=round(float(np.mean(r == expected)), 3),
                     longest_run_s=int(r.max()),
                     onset_second_positive_share=round(onset_positive, 3)))
wr = pd.DataFrame(rows).set_index("label")
print(wr.to_string())
wr.to_csv(f"{CFG['outdir']}/rev_window_runs.csv")
for h in ["W5", "W60", "W300"]:
    assert wr.loc[h, "onset_second_positive_share"] == 0.0, f"{h}: onset second labelled positive"
assert wr.loc["D5", "onset_second_positive_share"] == 1.0, "D5 does not keep the onset second"
print("\nWarning windows never include the onset second; D5 always does.")

## 4. Label variants for the sensitivity analysis

* **Primary**: the stored labels.
* **A, unmatched removed**: the unmatched June onsets are dropped, so their windows and their onset-second exclusions disappear.
* **B, unmatched imputed**: each unmatched episode is given the median matched duration of its own message over the full log (the overall median where a message has no matched episode), truncated one second before the next onset of the same fault and at the six-hour cap.

The variants are written to `labels_variants_june.pkl`, aligned row for row with the labelled telemetry, for `10_UNMATCHED` to read.

In [ ]:
matched = ep[ep.matched]
med_by_fault = matched.groupby("fault")["duration_s"].median()
med_all = matched["duration_s"].median()

def impute(e):
    e = e.copy()
    e["next_on"] = e.sort_values("onset").groupby(["cluster", "fault"])["onset"].shift(-1)
    fill = e.fault.map(med_by_fault).fillna(med_all)
    cap = ((e.next_on - e.onset).dt.total_seconds() - 1).fillna(np.inf).clip(lower=0)
    dur = np.minimum(np.minimum(fill, cap), CFG["max_episode_s"])
    u = ~e.matched
    e.loc[u, "duration_s"] = dur[u]
    e.loc[u, "recovery"] = e.loc[u, "onset"] + pd.to_timedelta(dur[u], unit="s")
    return e.drop(columns="next_on")

ep_B = impute(ep)
variants = {"A": span[span.matched], "B": ep_B[ep_B.june]}
print(f"Imputed durations, June unmatched: median {ep_B[ep_B.june & ~ep_B.matched].duration_s.median():.0f} s, "
      f"total {ep_B[ep_B.june & ~ep_B.matched].duration_s.sum():.0f} s")

out = pd.DataFrame({"DATETIME": tel["DATETIME"].values})
rows = []
for h, (pre, post, kind) in CFG["horizons"].items():
    rows.append(dict(label=h, variant="primary", positive=int((tel[h] == 1).sum()),
                     excluded=int((tel[h] == -1).sum())))
    for v, e in variants.items():
        y, _ = label(idx, e, pre, post, kind)
        out[f"{h}_{v}"] = y
        rows.append(dict(label=h, variant=v, positive=int((y == 1).sum()),
                         excluded=int((y == -1).sum())))
var = pd.DataFrame(rows).pivot(index="label", columns="variant", values=["positive", "excluded"])
print(var.to_string())
var.to_csv(f"{CFG['outdir']}/rev_label_variants.csv")
out.to_pickle(f"{CFG['builddir']}/labels_variants_june_v4.pkl")
del tel, out; _ = gc.collect()

## 5. July labels

The ingested telemetry runs to 12 July 2025. The 1 to 12 July records are labelled exactly as June: the same episodes, the same windows and the same exclusion, using onsets that fall within the period.

In [ ]:
raw = pd.read_pickle(CFG["raw_v2"])
k0, k1 = CFG["july"]
jul = raw[(raw.DATETIME >= k0) & (raw.DATETIME <= k1)].copy()
del raw; gc.collect()
jul = jul.sort_values("DATETIME").reset_index(drop=True)
jidx = jul["DATETIME"].values
jspan = ep[ep.onset.between(pd.Timestamp(k0), pd.Timestamp(k1))]

rows = []
for h, (pre, post, kind) in CFG["horizons"].items():
    y, _ = label(jidx, jspan, pre, post, kind)
    jul[h] = y
    n1, n0 = int((y == 1).sum()), int((y == 0).sum())
    rows.append(dict(label=h, positive=n1, excluded=int((y == -1).sum()), negative=n0,
                     pos_pct=round(100 * n1 / (n1 + n0), 3)))
jl = pd.DataFrame(rows).set_index("label")
days = (pd.Timestamp(k1) - pd.Timestamp(k0)).total_seconds() + 1
print(f"July records: {len(jul):,} ({100*len(jul)/days:.1f}% of the 12 days)")
print(f"July onsets: {len(jspan)}, blower {int(jspan.group.eq('blower').sum())} "
      f"({100*jspan.group.eq('blower').mean():.1f}%), unmatched {int((~jspan.matched).sum())}")
print(jl.to_string())
jl.to_csv(f"{CFG['outdir']}/rev_july_labels.csv")
jul.to_pickle(f"{CFG['builddir']}/telemetry_labelled_july_v4.pkl")
SUMMARY.update(july_records=len(jul), july_onsets=len(jspan),
               july_blower_share_pct=round(100 * jspan.group.eq("blower").mean(), 1))
del jul; _ = gc.collect()

## Summary

In [ ]:
s = pd.Series(SUMMARY, name="value")
print(s.to_string())
s.to_csv(f"{CFG['outdir']}/rev_labels_summary.csv")

___
**END**